In [2]:
%pip install anthropic python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
     ---------------------------------------- 0.0/109.4 kB ? eta -:--:--
     --- ------------------------------------ 10.2/109.4 kB ? eta -:--:--
     ---------- ---------------------------- 30.7/109.4 kB 1.3 MB/s eta 0:00:01
     -------------------- ---------------- 61.4/109.4 kB 544.7 kB/s eta 0:00:01
     ------------------------------- ----- 92.2/109.4 kB 655.4 kB/s eta 0:00:01
     ------------------------------------ 109.4/109.4 kB 527.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/956.9 kB ? eta -:--:--
   --- ------------------------------------ 92.2/956.9 kB 2.6 MB/s eta 0:00:01
   ---- ----------------------------------- 112.6/956.9 kB 2.2 MB/s eta 0:00:01
   --------- ------------------------------ 225.3/956.9 kB 1.5 MB/s eta 0:00:01
   --------------- ------------------------ 368.6/956.9 kB 2.1 MB/s eta 0:00:01
   ----------------- ---------------------- 419.8/956.9 kB 1.9 MB/s eta 0:


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\umaha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
# Use environment variables already available to the kernel
import os

print("API key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

API key loaded: False


In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Load a local .env file if present in the notebook folder or its parent
for env_path in [Path.cwd() / ".env", Path.cwd().parent / ".env"]:
    if env_path.exists():
        load_dotenv(env_path, override=False)

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY is not set. Set it in your shell, VS Code environment, or add it to .env.")
if len(api_key) < 20:
    raise ValueError("ANTHROPIC_API_KEY looks too short to be valid.")

from anthropic import Anthropic

client = Anthropic(api_key=api_key)
model = "claude-sonnet-4-5"
print("Client ready")

Client ready


In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text


In [9]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)


RawMessageStartEvent(message=Message(id='msg_011Cd8nqjacfTxdpRuyikF8z', container=None, content=[], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=2, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A rel', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='ational database containing fabricated customer records with randomly generated names, addresses, phone', type='text_delta'), index=0, type='content_block_delta')
R

In [7]:
messages = []

add_user_message(messages, "Write a 1 sentence descripion of fake database.")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        #print(text, end="")

        pass

stream.get_final_message()  # Wait for the final message to be received



ParsedMessage(id='msg_011CdCZTQUoFkN6ZNdVdUF6r', container=None, content=[ParsedTextBlock(citations=None, text='A fake database is a simulated or mock data storage system that mimics the structure and behavior of a real database but contains fabricated, test, or placeholder data instead of actual production information.', type='text', parsed_output=None)], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=19, output_tokens=42, output_tokens_details=None, server_tool_use=None, service_tier='standard'))